In [ ]:
import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Hàm để lưu mô hình học máy, scaler và OneHotEncoder
def save_model_scaler_and_encoder(model, scaler, encoder, model_filename):
    """
    Lưu mô hình học máy, scaler và OneHotEncoder vào một file duy nhất.
    
    Parameters:
    - model: Mô hình học máy đã huấn luyện (ví dụ: RandomForestClassifier, LogisticRegression,...)
    - scaler: Scaler đã được huấn luyện (ví dụ: StandardScaler)
    - encoder: OneHotEncoder đã được huấn luyện
    - model_filename: Tên file để lưu mô hình, scaler và encoder.
    """
    # Lưu vào một file duy nhất (sử dụng joblib để lưu mô hình, scaler và encoder)
    joblib.dump((model, scaler, encoder), model_filename)
    print(f'Model, scaler, and encoder have been saved to {model_filename}')

# Ví dụ sử dụng
if __name__ == "__main__":
    # Load dữ liệu Iris
    data = load_iris()
    X = data.data
    y = data.target

    # Chia dữ liệu thành train và test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # OneHotEncoding cho các cột phân loại (ví dụ: giả sử cột 0 là phân loại)
    encoder = OneHotEncoder(sparse=False)
    # Chỉ cần one-hot cho các cột phân loại trong X (ví dụ, giả sử cột đầu tiên là phân loại)
    X_train_encoded = encoder.fit_transform(X_train[:, [0]])
    X_test_encoded = encoder.transform(X_test[:, [0]])
    
    # Dữ liệu X sau khi one-hot encoding
    X_train_encoded_full = np.hstack((X_train_encoded, X_train[:, 1:]))
    X_test_encoded_full = np.hstack((X_test_encoded, X_test[:, 1:]))

    # Tiền xử lý - Sử dụng StandardScaler để chuẩn hóa dữ liệu
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_encoded_full)
    X_test_scaled = scaler.transform(X_test_encoded_full)

    # Huấn luyện mô hình RandomForest
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Lưu mô hình, scaler và encoder vào một file duy nhất
    save_model_scaler_and_encoder(model, scaler, encoder, 'model_scaler_encoder.pkl')
Giải thích:
One-Hot Encoding:

Chúng ta sử dụng OneHotEncoder từ sklearn.preprocessing để chuyển đổi một hoặc nhiều cột phân loại thành dạng One-Hot. Trong ví dụ này, giả sử cột đầu tiên là phân loại.

sparse=False giúp OneHotEncoder trả về mảng NumPy thay vì ma trận sparse, giúp dễ dàng thao tác hơn trong một số trường hợp.

Scaler (StandardScaler):

StandardScaler được sử dụng để chuẩn hóa dữ liệu sau khi đã thực hiện One-Hot Encoding.

Mô hình học máy:

Sử dụng RandomForestClassifier làm mô hình học máy. Bạn có thể thay thế mô hình này bằng bất kỳ mô hình nào khác tuỳ theo bài toán.

Lưu mô hình, scaler và encoder:

Hàm save_model_scaler_and_encoder sẽ lưu tất cả các thành phần đã huấn luyện (mô hình, scaler và encoder) vào một file duy nhất.

Thư viện joblib được sử dụng để lưu trữ chúng dưới dạng một file pkl.

Cách tải lại mô hình, scaler và encoder:
Để tải lại mô hình, scaler và encoder từ file đã lưu, bạn có thể sử dụng hàm joblib.load() như sau:

python
Copy code
def load_model_scaler_and_encoder(filename):
    # Tải lại mô hình, scaler và encoder từ file
    model, scaler, encoder = joblib.load(filename)
    print(f'Model, scaler, and encoder have been loaded from {filename}')
    return model, scaler, encoder

# Ví dụ sử dụng:
if __name__ == "__main__":
    # Tải lại mô hình, scaler và encoder
    model, scaler, encoder = load_model_scaler_and_encoder('model_scaler_encoder.pkl')
    
    # Dùng mô hình và scaler đã tải lại để dự đoán
    X_new = [[5.1, 3.5, 1.4, 0.2]]  # Dữ liệu mẫu mới
    # Chuyển đổi dữ liệu mới qua One-Hot Encoding
    X_new_encoded = encoder.transform([[5.1]])  # Chỉ mã hóa cột phân loại đầu tiên
    X_new_full = np.hstack((X_new_encoded, X_new[:, 1:]))  # Thêm các cột còn lại
    X_new_scaled = scaler.transform(X_new_full)  # Chuẩn hóa dữ liệu mới
    prediction = model.predict(X_new_scaled)  # Dự đoán
    print(f'Prediction: {prediction}')